In [ ]:
# === 노트북 공통 preamble (llm_math 패키지 로드) ===
import sys
from pathlib import Path

# llm_math 패키지를 찾을 수 있는 후보 경로들
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 현재 디렉토리의 상위 디렉토리들도 후보에 추가 (notebooks/ 폴더에서 실행 시)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math import 시도
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[주의] llm_math 패키지 로드 실패: {e}")
    print("  GitHub 레포를 클론하고 colab_setup.sh를 실행하세요.")
# === preamble 끝 ===


# Ch 15. Multi-Head Attention — 다양한 시선으로 보기

> **학습 목표**
> - Multi-Head Attention이 왜 여러 헤드를 병렬로 사용하는지 이해한다
> - 헤드 수 $h$와 헤드 차원 $d_k = d/h$의 관계를 설명한다
> - MHA, MQA, GQA의 차이와 메모리·속도 효과를 비교한다

## 15.1 단일 헤드의 한계

하나의 어텐션 헤드는 한 가지 관계(예: 주어-동사)만 학습하기 어렵다. **여러 관계를 동시에 학습**하려면?

## 15.2 Multi-Head Attention (MHA)

$$\mathrm{MultiHead}(Q, K, V) = \mathrm{Concat}(\mathrm{head}_1, \ldots, \mathrm{head}_h) W^O$$
$$\mathrm{head}_i = \mathrm{Attention}(Q W_i^Q, K W_i^K, V W_i^V)$$

- $W_i^Q \in \mathbb{R}^{d \times d_k}$, $W_i^K \in \mathbb{R}^{d \times d_k}$, $W_i^V \in \mathbb{R}^{d \times d_v}$
- $W^O \in \mathbb{R}^{h d_v \times d}$
- 보통 $d_k = d_v = d/h$

예: $d = 512, h = 8$ → $d_k = 64$. 각 헤드는 64차원에서 어텐션.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        # Q, K, V, O 프로젝션 (한 번에)
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x, mask=None):
        B, T, D = x.shape
        # (B, T, 3D) -> 3개 (B, T, D)
        qkv = self.W_qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        # 헤드 차원 분할: (B, T, D) -> (B, T, H, d_k) -> (B, H, T, d_k)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        # 어텐션: (B, H, T, d_k) @ (B, H, d_k, T) -> (B, H, T, T)
        scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores + mask
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        # (B, H, T, d_k) -> (B, T, H, d_k) -> (B, T, D)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out), weights

# 테스트
d_model, n_heads = 64, 8
mha = MultiHeadAttention(d_model, n_heads)
x = torch.randn(2, 10, d_model)
out, weights = mha(x)
print(f"Input: {x.shape}")
print(f"Output: {out.shape}")
print(f"Attention Weight: {weights.shape} (B, H, T, T)")
print(f"Parameter Count: {sum(p.numel() for p in mha.parameters()):,}")


## 15.3 헤드 해석 가능성

서로 다른 헤드는 다른 관계를 학습하는 경향:
- 일부 헤드: 이전 토큰 참조 (positional head)
- 일부 헤드: 문법적 관계 (주어-동사)
- 일부 헤드: 의미론적 유사성


In [ ]:
# 헤드별 어텐션 패턴 시각화
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=32, n_heads=4)

# 간단한 시퀀스 (4 토큰)
x = torch.randn(1, 8, 32)
out, weights = mha(x)

# 각 헤드의 어텐션 패턴
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
tokens = [f'tok{i}' for i in range(8)]
for h, ax in enumerate(axes):
    im = ax.imshow(weights[0, h].detach().numpy(), cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'Head {h}')
    ax.set_xlabel('Key Position')
    ax.set_ylabel('Query Position')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig('../figures/ch15_heads_patterns.png', dpi=100, bbox_inches='tight')
plt.show()
print("각 Head가 다른 Attention Pattern을 Training한다 (다양한 관계 Training).")


## 15.4 Multi-Query Attention (MQA)

**문제**: MHA는 KV 캐시가 $O(L \cdot h \cdot d_k)$ → 추론 시 메모리 큼.

**MQA** (Shazeer, 2019): $Q$만 헤드별, $K, V$는 모든 헤드가 공유.

$$\mathrm{head}_i = \mathrm{Attention}(Q W_i^Q, K W^K, V W^V)$$

- 파라미터 절약, KV 캐시 1/h로 감소
- 품질은 약간 떨어지지만 추론 속도 대폭 향상

## 15.5 Grouped-Query Attention (GQA)

**GQA** (Ainslie et al., 2023): 중간 점.
- $h$개 Q 헤드, $g$개 KV 헤드 그룹 ($1 \leq g \leq h$)
- $g=1$: MQA, $g=h$: MHA
- **LLaMA-2의 선택** (보통 $g = h/8$)


In [ ]:
# MHA, MQA, GQA 구현 비교
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_q_heads, n_kv_heads):
        super().__init__()
        assert d_model % n_q_heads == 0
        self.d_model = d_model
        self.n_q_heads = n_q_heads
        self.n_kv_heads = n_kv_heads
        self.d_k = d_model // n_q_heads
        # Q: n_q_heads * d_k, K/V: n_kv_heads * d_k
        self.W_q = nn.Linear(d_model, n_q_heads * self.d_k, bias=False)
        self.W_k = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_v = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_o = nn.Linear(n_q_heads * self.d_k, d_model, bias=False)
        # 그룹당 Q 헤드 수
        self.n_rep = n_q_heads // n_kv_heads

    def forward(self, x, mask=None):
        B, T, D = x.shape
        q = self.W_q(x).view(B, T, self.n_q_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
        # KV를 Q 헤드 수만큼 반복 (group expansion)
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
        scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores + mask
        weights = F.softmax(scores, dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.W_o(out), weights

# 비교: MHA, GQA, MQA
d = 64
configs = [
    ('MHA (h=8)', 8, 8),
    ('GQA (g=4)', 8, 4),
    ('GQA (g=2)', 8, 2),
    ('MQA (g=1)', 8, 1),
]
print(f"{'Config':>12} | {'Params':>10} | {'KV size':>10}")
print("-" * 38)
for name, nq, nkv in configs:
    attn = GroupedQueryAttention(d, nq, nkv)
    n_params = sum(p.numel() for p in attn.parameters())
    kv_size = nkv * (d // nq)  # KV Head당 차원
    print(f"{name:>12} | {n_params:>10,} | {kv_size:>10}")


## 15.6 [CPU/GPU 벤치마크 ⑥] MHA vs MQA vs GQA

추론 시 KV 캐시 메모리와 속도 차이 비교.


In [ ]:
# MHA/GQA/MQA 추론 벤치마크
from llm_math.bench import time_fn

configs = [
    ('MHA', 8, 8),
    ('GQA-4', 8, 4),
    ('GQA-2', 8, 2),
    ('MQA', 8, 1),
]
B, T, D = 1, 256, 256
x = torch.randn(B, T, D)

print(f"\n{'Config':>8} | {'Params':>8} | {'CPU (ms)':>12} | {'GPU (ms)':>12}")
print("-" * 50)
for name, nq, nkv in configs:
    attn = GroupedQueryAttention(D, nq, nkv)
    n_params = sum(p.numel() for p in attn.parameters())
    res = time_fn(lambda x: attn(x), x, device='cpu', warmup=2, repeat=3)
    if torch.cuda.is_available():
        attn_gpu = GroupedQueryAttention(D, nq, nkv).cuda()
        x_gpu = x.cuda()
        res_gpu = time_fn(lambda x: attn_gpu(x), x_gpu, device='cuda', warmup=2, repeat=5)
        print(f"{name:>8} | {n_params:>8,} | {res['mean_ms']:>12.3f} | {res_gpu['mean_ms']:>12.3f}")
    else:
        print(f"{name:>8} | {n_params:>8,} | {res['mean_ms']:>12.3f} | {'N/A':>12}")


## 15.7 핵심 정리

| 변형 | Q 헤드 | KV 헤드 | KV 캐시 | 사용 |
|---|---|---|---|---|
| MHA | $h$ | $h$ | $O(Lhd_k)$ | 품질 최우선 |
| MQA | $h$ | 1 | $O(Ld_k)$ | 속도 최우선 |
| GQA | $h$ | $g$ | $O(Lgd_k)$ | LLaMA-2 표준 |

## 연습문제

1. MHA에서 헤드 수 1, 4, 8, 16으로 바꿔가며 학습 곡선을 비교하라.
2. 어텐션 헤드의 패턴을 시각화하고, "positional head" (대각선에 집중)가 있는지 확인하라.
3. MQA와 MHA의 파라미터 수 차이를 계산하고, 실험으로 검증하라.
4. GQA에서 $g = 1, 2, 4, 8$에 따른 추론 속도와 메모리를 비교하라.
5. Multi-Head Attention의 $W^O$ 행렬이 왜 필요한지 설명하라.

> 해설: `solutions/ch15_solutions.ipynb`
